# 06.16_Neuron_gene_emergecne_R

神经相关基因功能逐步涌现分析。

- 当前文件：`analysis/06_single_cell_analysis/06.16_Neuron_gene_emergecne_R.ipynb`
- 原始来源：`Codes/06.14_Neuron_gene_emergecne.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`Seurat`, `clusterProfiler`, `data.table`, `dplyr`, `ggplot2`, `readr`, `scales`, `stringr`, `tidyr`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


r_base

In [ ]:
fig_dir <- '/share/home/zhangze/zz/NeuralOrigin/Figures'

## Stepwise emergence of the neuronal gene function
神经元基因功能的逐步涌现

In [ ]:
library(Seurat)

### Step 1. 读取 RDS 并计算每个物种最宽松的 DEGs

In [ ]:
library(Seurat)
library(dplyr)

# 物种列表
species_list <- c("Dare", "Neve", "Clhe", "Auco", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")

# 输入/输出目录
work_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/"
out_dir  <- file.path("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step1_base_markers")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# 逐物种计算
for (sp in species_list) {
  message("Processing species (loose base markers): ", sp)

  # 读入 Seurat 对象
  obj <- readRDS(file.path(work_dir, paste0(sp, ".OG.normalized.rds")))
  Idents(obj) <- "CellTypes"

  # 最宽松差异分析
  sp_markers <- FindAllMarkers(
    obj,
    only.pos = TRUE,
    min.pct = 0,
    logfc.threshold = 0,
    test.use = "wilcox"
  )

  # 保存结果
  saveRDS(sp_markers, file = file.path(out_dir, paste0(sp, "_base_markers.rds")))
  write.csv(sp_markers, file = file.path(out_dir, paste0(sp, "_base_markers.csv")), row.names = FALSE)

  message("✅ Finished ", sp, " (", nrow(sp_markers), " genes)")
}

message("🎉 所有物种的最宽松 DEGs 已完成计算，保存在: ", out_dir)


### Step 2. 提取每个物种同一参数下 DEGs 并进行对照比较

In [ ]:
library(dplyr)
library(stringr)

species_list <- c("Dare", "Neve", "Clhe", "Auco", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")
base_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step1_base_markers"

minpct_vals <- seq(0.05, 0.30, by = 0.05)
logfc_vals  <- seq(0.20, 0.60, by = 0.05)

# 读入最宽松结果
base_markers <- lapply(species_list, function(sp) {
  readRDS(file.path(base_dir, paste0(sp, "_base_markers.rds")))
})
names(base_markers) <- species_list

results_clusters <- list()

for (minpct in minpct_vals) {
  for (lfc in logfc_vals) {
    # Dare 神经基因集
    dare_df <- base_markers[["Dare"]] %>%
      filter(
        avg_log2FC >= lfc,
        p_val_adj <= 0.05,
        (pct.1 >= minpct | pct.2 >= minpct),
        # str_detect(cluster, "Neuron|Neural")
        str_detect(cluster, "Neural")
      )
    dare_neuron_genes <- unique(dare_df$gene)

    print("-------------------------")
    print(minpct)
    print(lfc)
    print(length(dare_neuron_genes))
    
    for (sp in setdiff(species_list, "Dare")) {
      sp_df <- base_markers[[sp]] %>%
        filter(
          avg_log2FC >= lfc,
          p_val_adj <= 0.05,
          (pct.1 >= minpct | pct.2 >= minpct)
        )
      
      # 统计每个 cluster 命中的数量
      cluster_stats <- sp_df %>%
        filter(gene %in% dare_neuron_genes) %>%
        group_by(cluster) %>%
        summarise(
          n_genes = n_distinct(gene),
          genes   = paste(unique(gene), collapse = ";"),
          .groups = "drop"
        ) %>%
        mutate(
          species = sp,
          minpct = minpct,
          logfc  = lfc
        ) %>%
        select(minpct, logfc, species, cluster, n_genes, genes)
      
      results_clusters[[length(results_clusters)+1]] <- cluster_stats
    }
  }
}

# 合并结果
final_clusters <- bind_rows(results_clusters)

# 保存
out_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence"
write.csv(final_clusters, file.path(out_dir, "DareNeuron_vs_Others_byCluster.csv"), row.names = FALSE)

message("✅ 统计完成：已输出 Dare 神经 DEGs 在其他物种各细胞类型的分布。")


In [ ]:
# 导出到txt文件（每行一个基因）
length(dare_neuron_genes)
writeLines(dare_neuron_genes, "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step3_gene_function/Dare_NeuronOGs_minpct0.3_logfc0.6.txt")

### Step 3. 统计结果可视化

#### 3.1 固定参数下Dare DEGs在其它物种细胞类型中的分布情况

In [ ]:
library(dplyr)
library(ggplot2)

# 指定想要绘制的参数组合
target_minpct <- 0.3
target_logfc  <- 0.6

# 物种顺序
species_order <- c("Spla", "ClH23", "HoH13", "TrH2", "TrH1", "Auco", "Clhe", "Neve")

# 筛选目标参数的数据
plot_df <- final_clusters %>%
  filter(minpct == target_minpct, logfc == target_logfc) %>%
  mutate(
    species = factor(species, levels = species_order),
    cluster = factor(cluster)
  )

# 绘制柱状图（每个物种一个panel，横轴cluster，纵轴Dare神经基因数）
p <- ggplot(plot_df, aes(x = cluster, y = n_genes, fill = cluster)) +
  geom_col() +
  facet_wrap(~ species, ncol = 4, scales = "free_x") +
  theme_bw(base_size = 12) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  ) +
  labs(
    title = paste0("Dare Neural DEGs vs Other Species (minpct=", target_minpct, ", logfc=", target_logfc, ")"),
    x = "CellTypes (Cluster)",
    y = "Gene Number with Dare"
  )

# 显示图片
print(p)

# 保存图片
ggsave(
  filename = paste0(fig_dir, "/65.Bar.Dare_neural_based_Celltypes_all_minpct", target_minpct, "_logfc", target_logfc, ".png"),
  plot = p,
  width = 10,
  height = 8,
  dpi = 300,
  bg = "white"
)

# 也可以保存为PDF格式（矢量图，适合论文）
ggsave(
  filename = paste0(fig_dir, "/65.Bar.Dare_neural_based_Celltypes_all_minpct", target_minpct, "_logfc", target_logfc, ".pdf"),
  plot = p,
  width = 10,
  height = 8,
  bg = "white"
)

message("图片已保存")

#### 3.2 固定参数下Dare DEGs在其它物种堆叠柱状图（神经-非神经）

In [ ]:
library(dplyr)
library(ggplot2)

# 指定参数组合
target_minpct <- 0.3
target_logfc  <- 0.6

# 物种顺序
species_order <- c("Spla", "ClH23", "HoH13", "TrH2", "TrH1", "Auco", "Clhe", "Neve", "Dare")

# 定义 Neuronal / NonNeuronal 分类规则
is_neuronal <- function(cluster_name) {
  grepl("NPC|Neural|neuronal|peptidergic|Neuroid", cluster_name, ignore.case = TRUE)
}

# 筛选目标参数，并分类
plot_df2 <- final_clusters %>%
  filter(minpct == target_minpct, logfc == target_logfc) %>%
  mutate(
    cell_class = ifelse(is_neuronal(cluster), "Neuronal", "NonNeuronal"),
    species = factor(species, levels = species_order)
  ) %>%
  group_by(species, cell_class) %>%
  summarise(
    n_genes = sum(n_genes),
    .groups = "drop"
  ) %>%
  group_by(species) %>%
  mutate(
    proportion = n_genes / sum(n_genes)
  )

# 绘制堆叠柱状图（按比例）
p <- ggplot(plot_df2, aes(x = species, y = proportion, fill = cell_class)) +
  geom_col(position = "fill") +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  theme_bw(base_size = 12) +
  labs(
    title = paste0("Dare Neural DEGs vs Other Species (minpct=", target_minpct, ", logfc=", target_logfc, ")"),
    x = "Species",
    y = "Proportion of Genes",
    fill = "Cell Class"
  )

# 显示图片
print(p)

# 保存图片
ggsave(
  filename = paste0(fig_dir, "/66.Bar.Dare_neural_based_Celltypes_Neuronal_minpct", target_minpct, "_logfc", target_logfc, ".png"),
  plot = p,
  width = 10,
  height = 6,
  dpi = 300,
  bg = "white"
)

# 也可以保存为PDF格式（矢量图，适合论文）
ggsave(
  filename = paste0(fig_dir, "/66.Bar.Dare_neural_based_Celltypes_Neuronal_minpct", target_minpct, "_logfc", target_logfc, ".pdf"),
  plot = p,
  width = 10,
  height = 6,
  bg = "white"
)

message("图片已保存")

In [ ]:
plot_df2

#### 3.3 固定参数下Dare DEGs在其它物种折线图（神经）

In [ ]:
library(dplyr)
library(ggplot2)

# 指定参数组合
target_minpct <- 0.3
target_logfc  <- 0.6

# 物种顺序
species_order <- c("Spla", "ClH23", "HoH13", "TrH2", "TrH1", "Auco", "Clhe", "Neve")
species_colors <- c(
    "Spla"="#fba414", "ClH23"="#ffa8a7", "HoH13"="#eb7f7f", 
    "TrH2"="#ff5d4e", "TrH1"="#EC2B24", "Auco"="#2A52BE", 
    "Clhe"="#4374B3", "Neve"="#6DA0E2", "Dare"="#43b244"
)

# 定义 Neuronal 分类规则
is_neuronal <- function(cluster_name) {
  grepl("NPC|Neural|neuronal|peptidergic|Neuroid", cluster_name, ignore.case = TRUE)
}

# 数据准备：计算 Neuronal 占比
plot_df_line <- final_clusters %>%
  filter(minpct == target_minpct, logfc == target_logfc) %>%
  mutate(
    cell_class = ifelse(is_neuronal(cluster), "Neuronal", "NonNeuronal"),
    species = factor(species, levels = species_order)
  ) %>%
  group_by(species, cell_class) %>%
  summarise(n_genes = sum(n_genes), .groups = "drop") %>%
  group_by(species) %>%
  mutate(proportion = n_genes / sum(n_genes)) %>%
  filter(cell_class == "Neuronal")

# 绘制折线图 + 数值标注
p <- ggplot(plot_df_line, aes(x = species, y = proportion, group = 1)) +
  # 黑色折线
  geom_line(size = 1.2, color = "black") +
  # 按物种颜色的点
  geom_point(aes(color = species), size = 3) +
  # 数值标注
  geom_text(aes(label = scales::percent(proportion, accuracy = 1)),
            vjust = -0.5, color = "black", size = 3.5) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  scale_color_manual(values = species_colors) +  # 点的颜色使用species_colors
  theme_bw(base_size = 12) +
  theme(
    legend.position = "none"  # 隐藏图例，因为x轴已经显示物种
  ) +
  labs(
    title = paste0("Proportion of Dare Neural DEGs in Neuronal Cells Across Species (minpct=", 
                   target_minpct, ", logfc=", target_logfc, ")"),
    x = "Species",
    y = "Neuronal Proportion"
  )

# 显示图片
print(p)

# 保存图片
ggsave(
  filename = paste0(fig_dir, "/67.Line.Dare_neural_based_OGs_minpct", target_minpct, "_logfc", target_logfc, ".png"),
  plot = p,
  width = 10,
  height = 6,
  dpi = 300,
  bg = "white"
)

# 也可以保存为PDF格式（矢量图，适合论文）
ggsave(
  filename = paste0(fig_dir, "/67.Line.Dare_neural_based_OGs_minpct", target_minpct, "_logfc", target_logfc, ".pdf"),
  plot = p,
  width = 10,
  height = 6,
  bg = "white"
)

message("图片已保存")

#### 3.4 所有参数下Dare DEGs在其它物种折线图（神经）

In [ ]:
library(dplyr)
library(ggplot2)

# 物种顺序
species_order <- c("Spla", "ClH23", "HoH13", "TrH2", "TrH1", "Auco", "Clhe", "Neve")

# Neuronal 分类函数
is_neuronal <- function(cluster_name) {
  grepl("NPC|Neural|neuronal|peptidergic|Neuroid", cluster_name, ignore.case = TRUE)
}

# 计算所有参数下的 Neuronal 占比
plot_df_all <- final_clusters %>%
  mutate(
    cell_class = ifelse(is_neuronal(cluster), "Neuronal", "NonNeuronal"),
    species = factor(species, levels = species_order)
  ) %>%
  group_by(species, minpct, logfc, cell_class) %>%
  summarise(n_genes = sum(n_genes), .groups = "drop") %>%
  group_by(species, minpct, logfc) %>%
  mutate(proportion = n_genes / sum(n_genes)) %>%
  filter(cell_class == "Neuronal")

# 绘制折线图（54 条线）
p <- ggplot(plot_df_all, aes(x = species, y = proportion, 
                        group = interaction(minpct, logfc),
                        color = factor(logfc), linetype = factor(minpct))) +
  geom_line(size = 0.8, alpha = 0.7) +
  geom_point(size = 1) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  theme_bw(base_size = 12) +
  labs(
    title = "Proportion of Dare Neural DEGs in Neuronal Cells Across Species",
    subtitle = "54 parameter combinations (logfc colored, minpct line type)",
    x = "Species",
    y = "Neuronal Proportion",
    color = "logFC",
    linetype = "min.pct"
  )

# 显示图片
print(p)

# 保存图片
ggsave(
  filename = paste0(fig_dir, "/68.Line.Dare_neural_based_OGs_all_minpct", target_minpct, "_logfc", target_logfc, ".png"),
  plot = p,
  width = 10,
  height = 6,
  dpi = 300,
  bg = "white"
)

# 也可以保存为PDF格式（矢量图，适合论文）
ggsave(
  filename = paste0(fig_dir, "/68.Line.Dare_neural_based_OGs_all_minpct", target_minpct, "_logfc", target_logfc, ".pdf"),
  plot = p,
  width = 10,
  height = 6,
  bg = "white"
)

message("图片已保存")

#### 3.5 所有参数下Dare DEGs在其它物种箱型图（神经）

In [ ]:
library(dplyr)
library(ggplot2)

species_colors <- c(
    "Spla"="#fba414", "ClH23"="#ffa8a7", "HoH13"="#eb7f7f", 
    "TrH2"="#ff5d4e", "TrH1"="#EC2B24", "Auco"="#2A52BE", 
    "Clhe"="#4374B3", "Neve"="#6DA0E2", "Dare"="#43b244"
)

# 基于已有的 plot_df_all 直接绘制箱型图
p <- ggplot(plot_df_all, aes(x = species, y = proportion, fill = species)) +
  geom_boxplot(outlier.alpha = 0.3, width = 0.6) +
  stat_summary(fun = median, geom = "point", shape = 21, size = 2, 
               color = "black", fill = "yellow") +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  scale_fill_manual(values = species_colors) +  # 应用自定义颜色
  theme_bw(base_size = 12) +
  # theme(legend.position = "none") +
  labs(
    title = "Distribution of Dare Neural DEGs Proportion Across Species",
    subtitle = "54 parameter combinations per species",
    x = "Species",
    y = "Neuronal Proportion"
  )

# 显示图片
print(p)

# 保存图片
ggsave(
  filename = paste0(fig_dir, "/69.BoxPlot.Dare_neural_based_OGs_all_minpct", target_minpct, "_logfc", target_logfc, ".png"),
  plot = p,
  width = 10,
  height = 6,
  dpi = 300,
  bg = "white"
)

# 也可以保存为PDF格式（矢量图，适合论文）
ggsave(
  filename = paste0(fig_dir, "/69.BoxPlot.Dare_neural_based_OGs_all_minpct", target_minpct, "_logfc", target_logfc, ".pdf"),
  plot = p,
  width = 10,
  height = 6,
  bg = "white"
)

message("图片已保存")

#### 3.6 三大类物种中的占比箱型图（神经）

In [ ]:
library(dplyr)
library(ggplot2)

# 物种分组
species_group <- data.frame(
  species = c("Spla", "ClH23", "HoH13", "TrH2", "TrH1", "Clhe", "Auco", "Neve"),
  group   = c("Porifera", "Placozoa", "Placozoa", "Placozoa", "Placozoa",
              "Cnidaria", "Cnidaria", "Cnidaria")
)

# 加入 group
plot_df_group <- plot_df_all %>%
  left_join(species_group, by = "species")

# 绘制箱型图
ggplot(plot_df_group, aes(x = group, y = proportion, fill = group)) +
  geom_boxplot(outlier.alpha = 0.3, width = 0.6) +
  stat_summary(fun = median, geom = "point", shape = 21, size = 2, color = "black", fill = "yellow") +
  theme_bw(base_size = 12) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  labs(
    title = "Proportion of Dare Neural DEGs in Neuronal Cells Across Major Clades",
    subtitle = "54 parameter combinations",
    x = "Clade",
    y = "Neuronal Proportion"
  )


#### 3.7 三大类物种中的占比箱型图（神经）(添加中位线)

In [ ]:
library(dplyr)
library(ggplot2)

# 物种分组
species_group <- data.frame(
  species = c("Spla", "ClH23", "HoH13", "TrH2", "TrH1", "Clhe", "Auco", "Neve"),
  group   = c("Porifera", "Placozoa", "Placozoa", "Placozoa", "Placozoa",
              "Cnidaria", "Cnidaria", "Cnidaria")
)

phylum_colors <- c(
    "Porifera"="#fba414", "Placozoa"="#EC2B24", 
    "Cnidaria"="#2A52BE", "Bilateria"="#43b244"
)

# 加入 group，并固定顺序
plot_df_group <- plot_df_all %>%
  left_join(species_group, by = "species") %>%
  mutate(group = factor(group, levels = c("Porifera", "Placozoa", "Cnidaria")))

# 计算每个类群的中位数（用于标注）
median_df <- plot_df_group %>%
  group_by(group) %>%
  summarise(median_val = median(proportion), .groups = "drop")

# 绘制箱型图 + 中位数折线 + 数值标注
p <- ggplot(plot_df_group, aes(x = group, y = proportion, fill = group)) +
  geom_boxplot(outlier.alpha = 0.3, width = 0.6) +
  stat_summary(fun = median, geom = "line", aes(group = 1), 
               color = "black", size = 1.2) +
  stat_summary(fun = median, geom = "point", 
               color = "red", size = 3, shape = 21, fill = "yellow") +
  geom_text(data = median_df, aes(x = group, y = median_val, 
                                  label = scales::percent(median_val, accuracy = 1)),
            vjust = -1, color = "black", size = 4) +
  scale_fill_manual(values = phylum_colors) +  # 应用自定义颜色
  theme_bw(base_size = 12) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  labs(
    title = "Proportion of Dare Neural DEGs in Neuronal Cells Across Major Clades",
    subtitle = "Boxplots: 54 parameter combinations; Line & numbers: medians",
    x = "Clade",
    y = "Neuronal Proportion"
  )

# 显示图片
print(p)

# 保存图片
ggsave(
  filename = paste0(fig_dir, "/72.BoxPlot.Dare_neural_based_OGs_phylum_minpct", target_minpct, "_logfc", target_logfc, ".png"),
  plot = p,
  width = 6,
  height = 6,
  dpi = 300,
  bg = "white"
)

# 也可以保存为PDF格式（矢量图，适合论文）
ggsave(
  filename = paste0(fig_dir, "/72.BoxPlot.Dare_neural_based_OGs_phylum_minpct", target_minpct, "_logfc", target_logfc, ".pdf"),
  plot = p,
  width = 6,
  height = 6,
  bg = "white"
)

message("图片已保存")

### Step 4. 统计OG结果并绘制Venn

In [ ]:
library(dplyr)

# 输入文件目录
gene_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad"

# 输出目录
out_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step2_group_gene_lists"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# 物种分组（四大类）
groups <- list(
  Porifera = "Spla.OG.genes.txt",
  Placozoa = c("ClH23.OG.genes.txt", "HoH13.OG.genes.txt", 
               "TrH2.OG.genes.txt", "TrH1.OG.genes.txt"),
  Cnidaria = c("Auco.OG.genes.txt", "Clhe.OG.genes.txt", "Neve.OG.genes.txt"),
  Bilateria = "Dare.OG.genes.txt"
)

# 读取函数
read_genes <- function(file) {
  read.delim(file.path(gene_dir, file), header = FALSE, stringsAsFactors = FALSE)[[1]]
}

# 计算每个大类的基因集（并集 union）
group_genes <- lapply(groups, function(files) {
  gene_lists <- lapply(files, read_genes)
  Reduce(union, gene_lists)  # 取并集
})

# 输出每个大类的结果
for (grp in names(group_genes)) {
  genes <- unique(group_genes[[grp]])
  out_file <- file.path(out_dir, paste0(grp, "_genes.txt"))
  write.table(genes, file = out_file,
              row.names = FALSE, col.names = FALSE, quote = FALSE)
  message("✅ 输出完成: ", grp, " (", length(genes), " genes)")
}

# 保存汇总统计表
summary_df <- data.frame(
  Group = names(group_genes),
  N_Genes = sapply(group_genes, function(x) length(unique(x)))
)
write.csv(summary_df, file = file.path(out_dir, "Group_gene_counts.csv"), row.names = FALSE)

message("📊 汇总统计已保存: Group_gene_counts.csv")


### Step 5. 神经基因功能逐渐涌现

#### 5.0 提取Dare的OGs和genes

In [ ]:
colnames(base_markers[["Dare"]])

In [ ]:
# ===== 路径设置 =====
map_dir    <- "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup"

# 固定参数
target_minpct <- 0.3
target_logfc  <- 0.6

# ===== 核心函数 =====
# Step 1. 提取神经相关 OGs
dare_df <- base_markers[["Dare"]] %>%
  filter(
    avg_log2FC >= target_logfc,
    p_val_adj <= 0.05,
    (pct.1 >= target_minpct | pct.2 >= target_minpct),
    str_detect(cluster, "Neural")
  )
Dare_ogs <- unique(dare_df$gene)

# 保存 OGs
Dare_og_file <- file.path(
  "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step3_gene_function", 
  paste0("Dare_NeuronOGs_minpct", target_minpct, "_logfc", target_logfc, ".txt"))
write.table(Dare_ogs, Dare_og_file, row.names = FALSE, col.names = FALSE, quote = FALSE)
  
# Step 2. 映射基因
map_file <- file.path(map_dir, paste0("Dare.protein_to_orthogroup.csv"))
map_df <- read_csv(map_file, show_col_types = FALSE)
Dare_genes <- map_df %>%
  filter(orthogroup %in% Dare_ogs) %>%
  pull(protein_id) %>%
  unique()

if (length(Dare_genes) == 0) {
  message("⚠️ 没有映射到基因")
  return(NULL)
}
  
Dare_gene_file <- file.path(
  "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step3_gene_function",
  paste0("Dare_NeuronGenes_minpct", target_minpct, "_logfc", target_logfc, ".txt"))
write.table(Dare_genes, Dare_gene_file, row.names = FALSE, col.names = FALSE, quote = FALSE)

#### 5.1 统计每个物种的功能富集结果

In [ ]:
library(dplyr)
library(stringr)
library(readr)
library(data.table)
library(clusterProfiler)
library(ggplot2)

# ===== 路径设置 =====
neuron_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence"
map_dir    <- "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup"
egg_dir    <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/eggNOG_output_processed"

# 固定参数
target_minpct <- 0.3
target_logfc  <- 0.6

# 物种列表
species_list <- c("Spla","ClH23","HoH13","TrH2","TrH1","Auco","Clhe","Neve","Dare")

# 输出目录
out_dir <- file.path(neuron_dir, "Step3_gene_function")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# ===== 核心函数 =====
run_enrichment_for_species <- function(species, minpct, logfc, df) {
  message("=== 处理物种: ", species, " ===")
  
  # Step 1. 提取神经相关 OGs
  ogs <- df %>%
    filter(species == !!species,
           minpct == minpct,
           logfc == logfc,
           str_detect(cluster, "NPC|Neural|neuronal|peptidergic|Neuroid")) %>%
          #  str_detect(cluster, "Neural|neuronal|peptidergic|Neuroid")) %>%
    pull(genes) %>%
    str_split(";") %>%
    unlist() %>%
    unique()
  
  if (length(ogs) == 0) {
    message("⚠️ ", species, " 没有神经相关 OGs")
    return(NULL)
  }
  
  # 保存 OGs
  og_file <- file.path(out_dir, paste0(species, "_NeuronOGs_minpct", minpct, "_logfc", logfc, ".txt"))
  write.table(ogs, og_file, row.names = FALSE, col.names = FALSE, quote = FALSE)
  
  # Step 2. 映射基因
  map_file <- file.path(map_dir, paste0(species, ".protein_to_orthogroup.csv"))
  map_df <- read_csv(map_file, show_col_types = FALSE)
  genes <- map_df %>%
    filter(orthogroup %in% ogs) %>%
    pull(protein_id) %>%
    unique()
  
  if (length(genes) == 0) {
    message("⚠️ ", species, " 没有映射到基因")
    return(NULL)
  }
  
  gene_file <- file.path(out_dir, paste0(species, "_NeuronGenes_minpct", minpct, "_logfc", logfc, ".txt"))
  write.table(genes, gene_file, row.names = FALSE, col.names = FALSE, quote = FALSE)
  
  # Step 3. eggNOG 注释
  egg_file <- file.path(egg_dir, paste0(species, ".emapper.annotations.tsv"))
  egg <- fread(egg_file) %>% as.data.frame()
  egg[egg == ""] <- NA
  
  eggnog_lines_with_go <- !is.na(egg$GOs) & egg$GOs != ""
  eggnog_annotations_go <- str_split(egg[eggnog_lines_with_go, "GOs"], ",")
  gene_to_go <- data.frame(
    gene = rep(egg$query[eggnog_lines_with_go], times = sapply(eggnog_annotations_go, length)),
    term = unlist(eggnog_annotations_go)
  )
  term2gene <- gene_to_go[, c("term", "gene")]
  
  # Step 4. 富集分析
  ego <- enricher(gene = genes,
                  TERM2GENE = term2gene,
                  pvalueCutoff = 0.3,
                  pAdjustMethod = "BH")
  df_res <- as.data.frame(ego)
  
  # 保存原始结果
  # out_csv <- file.path(out_dir, paste0(species, "_GOenrich_minpct", minpct, "_logfc", logfc, ".csv"))
  # write.csv(df_res, out_csv, row.names = FALSE)
  
  # Step 5. 整理结果
  if (nrow(df_res) > 0) {
    df1 <- go2term(df_res$ID)
    df_res <- left_join(df_res, df1, by = c("ID" = "go_id"))
    df_res$term <- df_res$Term; df_res$Term <- NULL
    
    df2 <- go2ont(df_res$ID)
    df_res <- left_join(df_res, df2, by = c("ID" = "go_id"))
    df_res$Ont <- df_res$Ontology; df_res$Ontology <- NULL
    
    out_csv <- file.path(out_dir, paste0(species, "_GOenrich_minpct", minpct, "_logfc", logfc, ".csv"))
    write.csv(df_res, out_csv, row.names = FALSE)

    df3 <- df_res %>%
      select(term, Ont, pvalue, Count) %>%
      filter(!is.na(term) & !is.na(Ont) & pvalue < 0.05) %>%
      arrange(pvalue)
    
    df3_top <- df3 %>% slice_head(n = 10)
    
    # Step 6. 绘图
    p_barplot <- ggplot(df3_top, aes(x = reorder(term, -log10(pvalue)), y = -log10(pvalue))) +
      geom_col(aes(fill = Ont)) +
      coord_flip() +
      labs(x = "", y = "-log10(pvalue)",
           title = paste0("GO enrichment of ", species, " Neuron Genes (minpct=", minpct, ", logfc=", logfc, ")")) +
      theme_bw()
    ggsave(file.path(out_dir, paste0(species, "_GO_barplot.pdf")), p_barplot, width = 8, height = 5)
    
    p_dotplot <- ggplot(df3_top, aes(x = reorder(term, -log10(pvalue)), y = -log10(pvalue))) +
      geom_point(aes(size = Count, color = Ont)) +
      scale_size_continuous(range = c(3, 8)) +
      coord_flip() +
      labs(x = "", y = "-log10(pvalue)",
           title = paste0("GO enrichment of ", species, " Neuron Genes (minpct=", minpct, ", logfc=", logfc, ")")) +
      theme_bw()
    ggsave(file.path(out_dir, paste0(species, "_GO_dotplot.pdf")), p_dotplot, width = 8, height = 5)
  }
  
  message("✅ ", species, " 完成: ", length(ogs), " OGs → ", length(genes), " genes → ", nrow(df_res), " GO terms")
  return(df_res)
}

# ===== 批量运行 =====
df_all <- read.csv(file.path(neuron_dir, "DareNeuron_vs_Others_byCluster.csv"), stringsAsFactors = FALSE)
all_results <- lapply(species_list, function(sp) run_enrichment_for_species(sp, target_minpct, target_logfc, df_all))
names(all_results) <- species_list


In [ ]:
# 0.3,0.6,"ClH23","peptidergic",10,"OG0002183;OG0000133;OG0000967;OG0000700;OG0000132;OG0000785;OG0000198;OG0001481;OG0000020;OG0000135"
# 0.3,0.6,"HoH13","peptidergic",8,"OG0002183;OG0000700;OG0000128;OG0000132;OG0000967;OG0000198;OG0000054;OG0001418"
# 0.3,0.6,"TrH2","peptidergic",3,"OG0002430;OG0002183;OG0000132"
# 0.3,0.6,"TrH1","peptidergic",7,"OG0002183;OG0000785;OG0000700;OG0000658;OG0002013;OG0000967;OG0000128"

# 0.3,0.6,"Auco","NE, Neural cell",6,"OG0000056;OG0000166;OG0000260;OG0000574;OG0000967;OG0000198"
# 0.3,0.6,"Clhe","Neural",4,"OG0000056;OG0000166;OG0000011;OG0000054"
# 0.3,0.6,"Neve","neuronal",11,"OG0000658;OG0000166;OG0006047;OG0000984;OG0000246;OG0002430;OG0002183;OG0000967;OG0000054;OG0005716;OG0000128"


#### 5.2 跨物种 GO term 共享性统计 + 热图绘制

In [ ]:
library(dplyr)
library(readr)

# 路径
go_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step3_gene_function"
out_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence"
species_list <- c("Spla","ClH23","HoH13","TrH2","TrH1","Auco","Clhe","Neve")

# 读取并合并
all_go <- lapply(species_list, function(sp) {
  file <- file.path(go_dir, paste0(sp, "_GOenrich_minpct0.3_logfc0.6.csv"))
  if (file.exists(file)) {
    df <- read.csv(file, stringsAsFactors = FALSE)
    df$species <- sp   # 加物种标签
    return(df)
  } else {
    warning("⚠️ 文件不存在: ", file)
    return(NULL)
  }
})
all_go <- bind_rows(all_go)

# 提取关键信息
all_go_clean <- all_go %>%
  select(species, ID, term, Ont, GeneRatio, FoldEnrichment, pvalue, p.adjust, Count) %>%
  rename(GO_ID = ID, Term = term)

# 保存合并表
write.csv(all_go_clean,
          # file.path(out_dir, "AllSpecies_GOenrich_summary.csv"),
          file.path(out_dir, "AllSpecies_GOenrich_summary_noNPC.csv"),
          row.names = FALSE)

message("✅ 合并完成: ", length(unique(all_go_clean$species)),
        " 个物种，", length(unique(all_go_clean$GO_ID)), " 个 GO term")


In [ ]:
library(dplyr)
library(ggplot2)
library(tidyr)

# ===== 1. 读取合并后的结果表 =====
all_go <- read.csv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/AllSpecies_GOenrich_summary.csv",
# all_go <- read.csv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/AllSpecies_GOenrich_summary_noNPC.csv",
                   stringsAsFactors = FALSE)

# ===== 2. 过滤显著条目，并去除 NA 描述 =====
sig_go <- all_go %>%
  filter(p.adjust < 0.05, !is.na(Term), Term != "")

# ===== 3. 统计每个 GO term 在多少个物种显著 =====
go_freq <- sig_go %>%
  group_by(GO_ID, Term) %>%
  summarise(n_species = n_distinct(species), .groups = "drop") %>%
  arrange(desc(n_species))

# 选择跨物种出现频率最高的前 10 个 term
top_terms <- go_freq %>%
  slice_head(n = 10) %>%
  mutate(TermLabel = paste0(GO_ID, " (", Term, ")")) %>%
  pull(TermLabel)

# ===== 4. 准备绘图数据 =====
plot_df <- sig_go %>%
  mutate(TermLabel = paste0(GO_ID, " (", Term, ")"),
         logP = -log10(p.adjust)) %>%
  filter(TermLabel %in% top_terms)

# 设定物种顺序（避免乱序）
species_order <- c("Spla","ClH23","HoH13","TrH2","TrH1","Auco","Clhe","Neve")
plot_df$species <- factor(plot_df$species, levels = species_order)

# ===== 5. 热图绘制 =====
p_heatmap <- ggplot(plot_df, aes(x = species, y = TermLabel, fill = logP)) +
  geom_tile(color = "white") + 
  scale_fill_gradient(low = "white", high = "red") +
  theme_bw(base_size = 12) +
  labs(title = "Cross-species GO enrichment (Top shared terms)",
       x = "Species", y = "GO Term", fill = "-log10(adj.p)")
ggsave("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step4_GO_cross_species/CrossSpecies_GO_Heatmap.pdf",
# ggsave("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step4_GO_cross_species/CrossSpecies_GO_Heatmap_noNPC.pdf",
       p_heatmap, width = 12, height = 6)

# ===== 6. 气泡图绘制 =====
p_bubble <- ggplot(plot_df, aes(x = species, y = TermLabel)) +
  geom_point(aes(size = Count, color = logP)) +
  scale_color_gradient(low = "white", high = "red") +
  theme_bw(base_size = 12) +
  labs(title = "Cross-species GO enrichment bubble plot",
       x = "Species", y = "GO Term", size = "Gene Count", color = "-log10(adj.p)")

ggsave("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step4_GO_cross_species/CrossSpecies_GO_Bubble.pdf",
# ggsave("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/NeuronEmergence/Step4_GO_cross_species/CrossSpecies_GO_Bubble_noNPC.pdf",
       p_bubble, width = 12, height = 6)
